# ENRE Regulatory Analysis — NLP on Argentine Electricity Regulation (2016–2025)

**Author:** Mariano Romero · [LinkedIn](https://www.linkedin.com/in/marianoromero23)  
**Tools:** BERTopic · SentenceTransformers · UMAP · HDBSCAN · pandas · matplotlib · PyVis  
**Data source:** InfoLEG — Normativa Nacional (datos.jus.gob.ar)

---

### Research Question
> *How did the regulatory priorities of Argentina's electricity sector change across the Cambiemos (2016–2019), Frente de Todos (2019–2023), and La Libertad Avanza (2023–) administrations?*

### Pipeline
1. Download full InfoLEG normativa dataset (~424k norms)
2. Filter to 1,873 ENRE resolutions (2016–2025)
3. Preprocess text and train BERTopic
4. Label and merge topics into 10 regulatory clusters
5. Recover outliers via lexical rules
6. Compare topic distributions across administrations (full term + 18-month window)
7. Lexical keyness analysis
8. Interactive network graph (government × regulated entity)


## 1. Install Dependencies

In [ ]:
# ── Celda 1 — Instalación de dependencias ──────────────────────────────────
# Ejecutar una sola vez al inicio de cada sesión.
# Si Colab pide reiniciar el runtime, reiniciar y saltar directamente a Celda 2.

!pip install bertopic sentence-transformers umap-learn hdbscan pyvis networkx -q
print("✓ Dependencias instaladas")


## 2. Download and Build Corpus

In [ ]:
# ── Celda 2 — Descarga del dataset InfoLEG y filtrado al corpus ENRE ────────
# Descarga ~150MB. Tiempo estimado: 2–4 minutos.

import requests, zipfile, io, pandas as pd

URL_ZIP = (
    "https://datos.jus.gob.ar/dataset/d9a963ea-8b1d-4ca3-9dd9-07a4773e8c23"
    "/resource/bf0ec116-ad4e-4572-a476-e57167a84403"
    "/download/base-infoleg-normativa-nacional.zip"
)

print("Descargando base InfoLEG (~150MB)...")
r = requests.get(URL_ZIP, timeout=180)
z = zipfile.ZipFile(io.BytesIO(r.content))
nombre_csv = [f for f in z.namelist() if f.endswith(".csv")][0]
df_infoleg = pd.read_csv(z.open(nombre_csv), encoding="latin-1", low_memory=False)
print(f"✓ Base completa: {len(df_infoleg):,} normas")

# ── Filtrar resoluciones del ENRE (2016–2025) ────────────────────────────────
df_enre = df_infoleg[
    df_infoleg["organismo_origen"] == "ENTE NACIONAL REGULADOR DE LA ELECTRICIDAD"
].copy()

df_enre["fecha_boletin"] = pd.to_datetime(df_enre["fecha_boletin"], errors="coerce")

df_resoluciones = df_enre[
    df_enre["tipo_norma"].str.contains("Resoluci", na=False) &
    df_enre["fecha_boletin"].dt.year.between(2016, 2025)
].copy()

print(f"✓ Resoluciones ENRE 2016–2025: {len(df_resoluciones)}")
print("\nDistribución por año:")
print(df_resoluciones["fecha_boletin"].dt.year.value_counts().sort_index().to_string())


## 3. Assign Governments and Build NLP Corpus

In [ ]:
# ── Celda 3 — Asignación de gobierno y construcción del campo texto_nlp ─────

def asignar_gobierno(fecha):
    if pd.isna(fecha):
        return "desconocido"
    if fecha < pd.Timestamp("2019-12-10"):
        return "Cambiemos (2016–2019)"
    elif fecha < pd.Timestamp("2023-12-10"):
        return "Frente de Todos (2019–2023)"
    else:
        return "La Libertad Avanza (2023–)"

df_resoluciones["gobierno"] = df_resoluciones["fecha_boletin"].apply(asignar_gobierno)

# Campo de texto combinado: sumario + texto resumido
df_resoluciones["texto_nlp"] = (
    df_resoluciones["titulo_sumario"].fillna("") + " " +
    df_resoluciones["texto_resumido"].fillna("")
)

print("Resoluciones por gobierno:")
print(df_resoluciones["gobierno"].value_counts().to_string())
print(f"\n✓ Corpus listo: {len(df_resoluciones)} resoluciones")

df_resoluciones.to_csv("enre_corpus.csv", index=False)
print("Guardado: enre_corpus.csv")


## 4. Text Preprocessing

In [ ]:
# ── Celda 4 — Limpieza y normalización del texto ────────────────────────────
# Elimina stopwords legales, números y caracteres especiales.
# Se conserva vocabulario sustantivo relevante para topic modeling.

import re, pandas as pd

df = pd.read_csv("enre_corpus.csv")

STOPWORDS = {
    "que", "del", "las", "los", "por", "con", "para", "como", "una", "uno",
    "este", "esta", "dicha", "dicho", "mediante", "conforme", "respecto",
    "toda", "todo", "debe", "deberá", "podrá", "será", "presente", "mismo",
    "misma", "cuanto", "tanto", "artículo", "resolución", "enre", "visto",
    "considerando", "asimismo", "dicho", "plazo", "fecha", "objeto", "caso",
    "acto", "ello", "cabo", "parte", "haber", "tener", "hace",
    "registrar", "comunicar", "archivar", "publicar", "notificar",
    "ente", "nacional", "regulador", "electricidad", "argentina"
}

def limpiar(texto):
    texto = str(texto).lower()
    texto = re.sub(r"[^a-záéíóúüñ\s]", " ", texto)
    palabras = [p for p in texto.split() if p not in STOPWORDS and len(p) > 3]
    return " ".join(palabras)

df["texto_limpio"] = df["texto_nlp"].apply(limpiar)
docs = df["texto_limpio"].tolist()

print(f"✓ Documentos preprocesados: {len(docs)}")
print(f"\nEjemplo (primeros 300 chars):")
print(docs[0][:300])


## 5. Train BERTopic Model

Uses `paraphrase-multilingual-MiniLM-L12-v2` for Spanish-language embeddings.  
UMAP reduces to 5 dimensions; HDBSCAN clusters with `min_cluster_size=15`.  
**Runtime: ~20–35 minutes on Colab free tier.**


In [ ]:
# ── Celda 5 — Entrenamiento BERTopic ────────────────────────────────────────
# Tiempo estimado: 20–35 minutos (el paso de embeddings es el cuello de botella).

from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=15,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

modelo_emb = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

topic_model = BERTopic(
    embedding_model=modelo_emb,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    top_n_words=10,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)
df["topico_id"] = topics

info = topic_model.get_topic_info()
print(f"\n✓ Tópicos descubiertos: {len(info) - 1}  (excluye outliers)")
print(f"  Outliers (tópico -1): {(df['topico_id'] == -1).sum()} resoluciones")
print("\n--- Top keywords por tópico ---")
for tid in info[info["Topic"] != -1]["Topic"].tolist():
    palabras = topic_model.get_topic(tid)
    kws = ", ".join([w for w, _ in palabras[:8]])
    count = info[info["Topic"] == tid]["Count"].values[0]
    print(f"  Tópico {tid:2d} (n={count:3d}): {kws}")


## 6. Topic Labeling, Merging, and Outlier Recovery

Raw BERTopic output (33 clusters) is consolidated into **10 interpretable regulatory categories** through:
- Manual supervision of keyword lists per topic
- Lexical rule-based subdivision of Topic 0 (conventional transport vs. renewable generation)
- Keyword rules to reclassify 163 of 186 outliers


In [ ]:
# ── Celda 6 — Etiquetado, fusión de tópicos y recuperación de outliers ───────

# 6A: Mapa de fusión — de 33 tópicos BERTopic a 10 categorías regulatorias
mapa_fusion = {}

grupos = {
    "Acceso y ampliación de transporte convencional"             : [0],
    "Regulación y sanciones Edenor/Edesur"                      : [1, 10],
    "Recursos administrativos"                                   : [2, 32],
    "Actos administrativos y procedimientos"                     : [3, 4, 6, 19],
    "Tarifas y facturación"                                      : [5, 7, 12, 13],
    "Remuneración a transportistas"                              : [8, 11, 14, 18, 28],
    "Regulación de distribuidoras y transportistas provinciales" : [9, 15, 20, 21, 22, 23, 24, 25, 26, 29, 30, 31],
    "Audiencias públicas"                                        : [16, 17],
    "Tasas de fiscalización"                                     : [27],
}

for etiqueta, ids in grupos.items():
    for tid in ids:
        mapa_fusion[tid] = etiqueta

df["topico_nombre"] = df["topico_id"].map(mapa_fusion).fillna("Sin clasificar")

# 6B: Subdivisión del Tópico 0 — transporte convencional vs. generación renovable
patron_renovables = "solar|eolic|fotovoltaic|parque|renovable|biomasa|hidro|viento"

mask_renovable    = (df["topico_id"] == 0) & (df["texto_resumido"].str.contains(patron_renovables, case=False, na=False))
mask_convencional = (df["topico_id"] == 0) & (~mask_renovable)

df.loc[mask_renovable,    "topico_nombre"] = "Conexión de generación renovable al SADI"
df.loc[mask_convencional, "topico_nombre"] = "Acceso y ampliación de transporte convencional"

# 6C: Recuperación de outliers via reglas léxicas
def reclasificar_outlier(texto):
    t = str(texto).upper()
    if any(x in t for x in ["EDENOR", "EDESUR"]):
        return "Regulación y sanciones Edenor/Edesur"
    if any(x in t for x in ["VALORES HORARIOS", "TRANSPORTISTA INDEPENDIENTE",
                             "YACYLEC", "LITSA", "TRANSBA", "TRANSNOA",
                             "TRANSCOMAHUE", "TRANSPA", "TRANSPORTEL"]):
        return "Remuneración a transportistas"
    if any(x in t for x in ["SANCIONAR", "MULTA", "INCUMPLIMIENTO",
                             "EDELAP", "EPEN", "EPEC", "DISTROCUYO"]):
        return "Regulación de distribuidoras y transportistas provinciales"
    if any(x in t for x in ["RECTIFICAR", "ERROR MATERIAL", "ESTATUTO",
                             "MODIFICAR", "DEROGAR"]):
        return "Actos administrativos y procedimientos"
    if any(x in t for x in ["SOLAR", "EOLICO", "EÓLICO", "FOTOVOLTAIC",
                             "PARQUE", "RENOVABLE"]):
        return "Conexión de generación renovable al SADI"
    return "Sin clasificar"

mask_sc = df["topico_nombre"] == "Sin clasificar"
df.loc[mask_sc, "topico_nombre"] = df.loc[mask_sc, "texto_resumido"].apply(reclasificar_outlier)

# 6D: Corrección del tópico Edenor/Edesur — eliminar mal clasificadas
def empresa_afectada(texto):
    t = str(texto).upper()
    tiene_edenor = "EDENOR" in t
    tiene_edesur = "EDESUR" in t
    if tiene_edenor and tiene_edesur: return "Ambas"
    elif tiene_edenor:                return "Edenor"
    elif tiene_edesur:                return "Edesur"
    else:                             return "Otra"

df["empresa"] = df["texto_resumido"].apply(empresa_afectada)

mask_mal = (df["topico_nombre"] == "Regulación y sanciones Edenor/Edesur") & (df["empresa"] == "Otra")
df.loc[mask_mal, "topico_nombre"] = "Regulación de distribuidoras y transportistas provinciales"

print("✓ Distribución de tópicos final:")
print(df["topico_nombre"].value_counts().to_string())
print(f"\nSin clasificar: {df['topico_nombre'].eq('Sin clasificar').sum()} resoluciones (<1.3%)")

df.to_csv("enre_final.csv", index=False)
print("\nGuardado: enre_final.csv")


## 7. Comparative Analysis — Full Term vs. First 18 Months

To control for the different durations of each administration, topic distributions are compared both over the full term and over a standardized 18-month window from each government's start date.


In [ ]:
# ── Celda 7 — Análisis comparativo por gobierno ─────────────────────────────

import pandas as pd

df = pd.read_csv("enre_final.csv")
df["fecha_boletin"] = pd.to_datetime(df["fecha_boletin"])
df_validos = df[df["topico_nombre"] != "Sin clasificar"].copy()

orden_gov = [
    "Cambiemos (2016–2019)",
    "Frente de Todos (2019–2023)",
    "La Libertad Avanza (2023–)"
]

inicios = {
    "Cambiemos (2016–2019)"        : pd.Timestamp("2015-12-10"),
    "Frente de Todos (2019–2023)"  : pd.Timestamp("2019-12-10"),
    "La Libertad Avanza (2023–)"   : pd.Timestamp("2023-12-10"),
}

def mes_gestion(row):
    inicio = inicios.get(row["gobierno"])
    if pd.isna(row["fecha_boletin"]) or inicio is None:
        return None
    return (row["fecha_boletin"].year - inicio.year) * 12 + \
           (row["fecha_boletin"].month - inicio.month)

df_validos["mes_gestion"] = df_validos.apply(mes_gestion, axis=1)

# Gestión completa
pivot = (df_validos
         .groupby(["gobierno", "topico_nombre"])
         .size()
         .unstack(fill_value=0)
         .reindex(orden_gov))
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

# Primeros 18 meses (ventana equivalente)
df_18 = df_validos[df_validos["mes_gestion"].between(0, 17)]
pivot_18 = (df_18
            .groupby(["gobierno", "topico_nombre"])
            .size()
            .unstack(fill_value=0)
            .reindex(orden_gov))
pivot_18_pct = pivot_18.div(pivot_18.sum(axis=1), axis=0) * 100

print("=== DISTRIBUCIÓN TEMÁTICA — Gestión completa (%) ===")
print(pivot_pct.round(1).T.to_string())
print("\n=== DISTRIBUCIÓN TEMÁTICA — Primeros 18 meses (%) ===")
print(pivot_18_pct.round(1).T.to_string())


## 8. Visualizations — Heatmap and Delta Charts

In [ ]:
# ── Celda 8 — Heatmaps + gráficos delta LLA vs. Cambiemos ──────────────────

import matplotlib.pyplot as plt

etiquetas_cortas = {
    "Acceso y ampliación de transporte convencional"               : "Transporte convencional",
    "Actos administrativos y procedimientos"                       : "Actos administrativos",
    "Audiencias públicas"                                          : "Audiencias públicas",
    "Conexión de generación renovable al SADI"                    : "Generación renovable",
    "Recursos administrativos"                                     : "Recursos admin.",
    "Regulación de distribuidoras y transportistas provinciales"   : "Distrib./transp. provinciales",
    "Regulación y sanciones Edenor/Edesur"                        : "Edenor/Edesur",
    "Remuneración a transportistas"                                : "Remuneración transport.",
    "Tarifas y facturación"                                        : "Tarifas y facturación",
    "Tasas de fiscalización"                                       : "Tasas fiscalización",
}

def renombrar(pivot):
    p = pivot.copy()
    p.columns = [etiquetas_cortas.get(c, c) for c in p.columns]
    return p

pivot_full_r = renombrar(pivot_pct)
pivot_18_r   = renombrar(pivot_18_pct)

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

for idx, (pivot_plot, titulo) in enumerate([
    (pivot_full_r, "Gestión completa"),
    (pivot_18_r,   "Primeros 18 meses")
]):
    # Heatmap
    ax_heat = axes[idx][0]
    im = ax_heat.imshow(pivot_plot.values, cmap="Blues", aspect="auto", vmin=0, vmax=35)
    ax_heat.set_xticks(range(len(pivot_plot.columns)))
    ax_heat.set_xticklabels(pivot_plot.columns, rotation=40, ha="right", fontsize=9)
    ax_heat.set_yticks(range(len(pivot_plot.index)))
    ax_heat.set_yticklabels([g.split(" (")[0] for g in pivot_plot.index], fontsize=10)
    for i in range(len(pivot_plot.index)):
        for j in range(len(pivot_plot.columns)):
            val = pivot_plot.values[i, j]
            ax_heat.text(j, i, f"{val:.1f}", ha="center", va="center",
                        fontsize=8, color="white" if val > 20 else "black")
    plt.colorbar(im, ax=ax_heat, label="%")
    ax_heat.set_title(f"Heatmap — {titulo}", fontsize=11, pad=8)

    # Delta LLA vs Cambiemos
    ax_delta = axes[idx][1]
    delta = (pivot_plot.loc["La Libertad Avanza (2023–)"] -
             pivot_plot.loc["Cambiemos (2016–2019)"]).sort_values()
    colores_bar = ["#1D9E75" if v > 0 else "#D85A30" for v in delta.values]
    ax_delta.barh(range(len(delta)), delta.values, color=colores_bar, height=0.6)
    ax_delta.set_yticks(range(len(delta)))
    ax_delta.set_yticklabels(delta.index, fontsize=9)
    ax_delta.axvline(0, color="black", linewidth=0.8)
    ax_delta.set_xlabel("Puntos porcentuales")
    ax_delta.set_title(f"Delta LLA vs Cambiemos — {titulo}", fontsize=11, pad=8)
    for i, v in enumerate(delta.values):
        ax_delta.text(v + (0.2 if v >= 0 else -0.2), i, f"{v:+.1f}",
                     va="center", ha="left" if v >= 0 else "right", fontsize=8.5)

plt.suptitle("Evolución temática regulatoria ENRE 2016–2025\n"
             "Comparación por gestión completa vs. primeros 18 meses",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("analisis_comparativo_enre.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Guardado: analisis_comparativo_enre.png")


## 9. Lexical Keyness Analysis

Tracks politically significant vocabulary across administrations — normalizing by total word count to enable fair comparison across governments of different durations.


In [ ]:
# ── Celda 9 — Análisis de vocabulario político (keyness) ────────────────────

import re, pandas as pd, matplotlib.pyplot as plt

df = pd.read_csv("enre_final.csv")
df["fecha_boletin"] = pd.to_datetime(df["fecha_boletin"])
df_validos = df[df["topico_nombre"] != "Sin clasificar"].copy()

inicios = {
    "Cambiemos (2016–2019)"        : pd.Timestamp("2015-12-10"),
    "Frente de Todos (2019–2023)"  : pd.Timestamp("2019-12-10"),
    "La Libertad Avanza (2023–)"   : pd.Timestamp("2023-12-10"),
}
orden_gov = list(inicios.keys())

def mes_gestion(row):
    inicio = inicios.get(row["gobierno"])
    if pd.isna(row["fecha_boletin"]) or inicio is None: return None
    return (row["fecha_boletin"].year - inicio.year) * 12 + \
           (row["fecha_boletin"].month - inicio.month)

df_validos["mes_gestion"] = df_validos.apply(mes_gestion, axis=1)
df_18 = df_validos[df_validos["mes_gestion"].between(0, 17)]

vocab_politico = [
    "aprobar", "sancionar", "multa", "incumplimiento",
    "tarifa", "tarifas", "subsidio", "costo",
    "usuarios", "residencial", "inversiones", "social",
]

def keyness(df_sub):
    resultados = {}
    for gov in orden_gov:
        textos = df_sub[df_sub["gobierno"] == gov]["texto_resumido"].fillna("").str.lower()
        texto_concat = " ".join(textos.tolist())
        total = len(texto_concat.split())
        conteos = {}
        for palabra in vocab_politico:
            n = len(re.findall(r'\b' + re.escape(palabra) + r'\b', texto_concat))
            conteos[palabra] = round(n / total * 10000, 2)
        resultados[gov.split(" (")[0]] = conteos
    return pd.DataFrame(resultados)

df_kw = keyness(df_18)
print("=== VOCABULARIO POLÍTICO — Primeros 18 meses (por 10.000 palabras) ===")
print(df_kw.to_string())

# Visualización
fig, ax = plt.subplots(figsize=(13, 6))
x = range(len(vocab_politico))
width = 0.25
colores = ["#378ADD", "#D85A30", "#6B3FA0"]

for i, (gov, color) in enumerate(zip(df_kw.columns, colores)):
    ax.bar([xi + i*width for xi in x], df_kw[gov].values,
           width=width, label=gov, color=color, alpha=0.85)

ax.set_xticks([xi + width for xi in x])
ax.set_xticklabels(vocab_politico, rotation=35, ha="right", fontsize=10)
ax.set_ylabel("Frecuencia por 10.000 palabras")
ax.set_title("Vocabulario político por gobierno — ENRE 2016–2025\n(primeros 18 meses)", fontsize=12)
ax.legend(fontsize=9)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("vocabulario_politico_enre.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Guardado: vocabulario_politico_enre.png")


## 10. Interactive Network Graph — Government × Regulated Entity

Bipartite network showing the volume of regulatory resolutions between each administration and the main regulated entities (Edenor / Edesur). Edge weight = number of resolutions.


In [ ]:
# ── Celda 10 — Grafo interactivo gobierno × empresa regulada ─────────────────

import pandas as pd, networkx as nx, IPython
from pyvis.network import Network

df = pd.read_csv("enre_final.csv")

df_net = df.dropna(subset=["gobierno", "empresa"]).copy()
df_net = df_net[df_net["empresa"].str.contains("Edenor|Edesur", case=False, na=False)]

conexiones = (df_net
              .groupby(["gobierno", "empresa"])
              .size()
              .reset_index(name="peso"))
conexiones = conexiones[conexiones["peso"] > 0]

G = nx.Graph()
for _, row in conexiones.iterrows():
    G.add_node(row["gobierno"], group="Gobierno", title="Gobierno", size=40, color="#FF5733")
    G.add_node(row["empresa"],  group="Empresa",  title=row["empresa"], size=25, color="#33C1FF")
    G.add_edge(row["gobierno"], row["empresa"],
               value=row["peso"], title=f"Resoluciones: {row['peso']}")

net = Network(
    notebook=True, cdn_resources="in_line",
    width="100%", height="600px",
    bgcolor="#222222", font_color="white"
)
net.force_atlas_2based()
net.from_nx(G)
net.show("red_distribuidoras.html")
IPython.display.HTML(filename="red_distribuidoras.html")


## 11. Outputs and Next Steps

### Files generated
| File | Description |
|---|---|
| `enre_corpus.csv` | 1,873 ENRE resolutions with government label and NLP text |
| `enre_final.csv` | Full dataset with topic assignments |
| `analisis_comparativo_enre.png` | Heatmap + delta charts (full term & 18-month window) |
| `vocabulario_politico_enre.png` | Keyness bar chart |
| `red_distribuidoras.html` | Interactive network graph |

### Key Findings
- **LLA governs through approval, not sanction** — "aprobar" frequency triples vs. Cambiemos; "sancionar" drops
- **Tariff resolutions triple under LLA** in the 18-month comparable window
- **Renewable energy connections are high under both FdT and LLA** — not concentrated under Cambiemos
- **Audiencias públicas in sustained decline** across all three administrations

### Limitations & Next Steps
- Full resolution text (`texto_original`) coverage is partial — InfoLEG scraping returns 403 errors on many records; topic modeling relies on summaries
- Extend analysis to ENARGAS for gas sector comparison
- Apply keyness over full text once scraping pipeline is resolved
- Potential working paper: 3–5 pages for SSRN or SAAP congress submission
